In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [19]:
# import os
# os.listdir('/kaggle/input/')
# train_data = pd.read_csv('...')
# test_data = pd.read_csv('...')

In [20]:
# 数据加载
train_data = pd.read_csv('train.csv')   
test_data = pd.read_csv('test.csv')

sample_submission = pd.read_csv('sample_submission.csv')
train_data.head()

,ID,fullAddress,postcode,country,outcode,latitude,longitude,bathrooms,bedrooms,floorAreaSqM,livingRooms,tenure,propertyType,currentEnergyRating,sale_month,sale_year,price
0,0,"38 Adelina Grove, London, E1 3AD",E1 3AD,England,E1,51.519406,-0.053261,NaN,3.0,80.0,1.0,Freehold,Semi-Detached House,C,1,1995,77000
1,1,"6 Cleveland Grove, London, E1 4XL",E1 4XL,England,E1,51.521261,-0.053384,2.0,4.0,110.0,1.0,Leasehold,Terrace Property,D,1,1995,89995
2,2,"65 Sanderstead Road, London, E10 7PW",E10 7PW,England,E10,51.569054,-0.034892,1.0,3.0,84.0,1.0,Freehold,Terrace Property,D,1,1995,59000
3,3,"5 Queenswood Gardens, London, E11 3SE",E11 3SE,England,E11,51.564212,0.026292,NaN,2.0,72.0,1.0,Leasehold,Purpose Built Flat,NaN,1,1995,51500
4,4,"12 Woodlands Road, London, E11 4RW",E11 4RW,England,E11,51.563430,0.006260,1.0,3.0,104.0,1.0,Freehold,Mid Terrace House,D,1,1995,63500


In [21]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266325 entries, 0 to 266324
Data columns (total 17 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   ID                   266325 non-null  int64  
 1   fullAddress          266325 non-null  object 
 2   postcode             266325 non-null  object 
 3   country              266325 non-null  object 
 4   outcode              266325 non-null  object 
 5   latitude             266325 non-null  float64
 6   longitude            266325 non-null  float64
 7   bathrooms            217846 non-null  float64
 8   bedrooms             241482 non-null  float64
 9   floorAreaSqM         252519 non-null  float64
 10  livingRooms          229285 non-null  float64
 11  tenure               260604 non-null  object 
 12  propertyType         265817 non-null  object 
 13  currentEnergyRating  209511 non-null  object 
 14  sale_month           266325 non-null  int64  
 15  sale_year        

In [22]:
train_data['tenure'].value_counts()

tenure
Leasehold    155910
Freehold     102096
Feudal         1906
Shared          692
Name: count, dtype: int64

In [23]:
train_data['propertyType'].value_counts()

propertyType
Purpose Built Flat        68726
Flat/Maisonette           61139
Mid Terrace House         45649
Converted Flat            32552
Semi-Detached House       20475
Terrace Property          15114
End Terrace House         13063
Detached House             6666
Terraced                    927
Bungalow Property           267
Semi-Detached Property      254
Semi-Detached Bungalow      232
Detached Bungalow           186
Detached Property           160
End Terrace Property        149
Mid Terrace Property        120
Mid Terrace Bungalow         67
Terraced Bungalow            38
End Terrace Bungalow         33
Name: count, dtype: int64

In [24]:
# 数据预处理与特征工程
def pre_pross(df):
    df = df.drop(columns = ['ID', 'country', 'postcode', 'outcode', 'fullAddress'])

    df['sale_time'] = df['sale_year'] * 12 + df['sale_month']
    df['monthSin'] = np.sin(2 * np.pi * (df['sale_month'] - 1) / 12)  # 捕捉周期性特点
    df['monthCos'] = np.cos(2 * np.pi * (df['sale_month'] - 1) / 12)

    df['energyRating'] = df['currentEnergyRating'].apply(
        lambda ch: ord(ch) - ord('A') + 1 if not pd.isna(ch) else np.nan
    )
    df['lnAreaSqM'] = df['floorAreaSqM'].apply(
        lambda x: np.log(x) if not pd.isna(x) else np.nan   # 数据右偏
    )
    df['disFromCenter'] = np.sqrt((df['latitude'] - 51.5072)**2 + (df['longitude'] - 0.1275)**2)
    
    df['isFreehold'] = df['tenure'].apply(lambda x: (1.0 if x == 'Freehold' else 0.0) if not pd.isna(x) else np.nan)
    property_high = ['Mid Terrace Property', 'Detached Property', 'Detached House', 'Semi-Detached Property', 
                      'Terraced', 'End Terrace Property', 'Terrace Property']
    df['isPropertyHigh'] = df['propertyType'].apply(lambda x: (1.0 if x in property_high else 0.0) if not pd.isna(x) else np.nan)
    
    df = df.drop(columns = ['sale_year', 'sale_month', 'currentEnergyRating', 'floorAreaSqM', 'tenure', 'propertyType'])

    return df

train_data = pre_pross(train_data)
test_data = pre_pross(test_data)

In [25]:
X = train_data.drop(columns = ['price'])
y = train_data['price']  # shape为(n,)
y = np.log(y)    # 数据右偏

X_test = test_data.copy().reindex(columns = X.columns)  # 保证顺序一致
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266325 entries, 0 to 266324
Data columns (total 13 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   latitude        266325 non-null  float64
 1   longitude       266325 non-null  float64
 2   bathrooms       217846 non-null  float64
 3   bedrooms        241482 non-null  float64
 4   livingRooms     229285 non-null  float64
 5   sale_time       266325 non-null  int64  
 6   monthSin        266325 non-null  float64
 7   monthCos        266325 non-null  float64
 8   energyRating    209511 non-null  float64
 9   lnAreaSqM       252519 non-null  float64
 10  disFromCenter   266325 non-null  float64
 11  isFreehold      260604 non-null  float64
 12  isPropertyHigh  265817 non-null  float64
dtypes: float64(12), int64(1)
memory usage: 26.4 MB


In [26]:
X_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16547 entries, 0 to 16546
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   latitude        16547 non-null  float64
 1   longitude       16547 non-null  float64
 2   bathrooms       13923 non-null  float64
 3   bedrooms        15172 non-null  float64
 4   livingRooms     14452 non-null  float64
 5   sale_time       16547 non-null  int64  
 6   monthSin        16547 non-null  float64
 7   monthCos        16547 non-null  float64
 8   energyRating    15050 non-null  float64
 9   lnAreaSqM       14541 non-null  float64
 10  disFromCenter   16547 non-null  float64
 11  isFreehold      15957 non-null  float64
 12  isPropertyHigh  16380 non-null  float64
dtypes: float64(12), int64(1)
memory usage: 1.6 MB


In [27]:
# MICE多重插补
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

imputer = IterativeImputer(
    max_iter = 50,
    random_state = 42,
    tol = 1e-5  # 精度
)
X_imputed = imputer.fit_transform(X)
X = pd.DataFrame(X_imputed, columns = X.columns)

X_test_imputed = imputer.transform(X_test)
X_test = pd.DataFrame(X_test_imputed, columns = X_test.columns)

X.isna().sum()

latitude          0
longitude         0
bathrooms         0
bedrooms          0
livingRooms       0
sale_time         0
monthSin          0
monthCos          0
energyRating      0
lnAreaSqM         0
disFromCenter     0
isFreehold        0
isPropertyHigh    0
dtype: int64

In [28]:
from sklearn.preprocessing import StandardScaler
# 只挑需要列进行归一化
cols_to_scale = ['latitude', 'longitude', 'disFromCenter', 'lnAreaSqM', 'energyRating',
                 'bathrooms', 'bedrooms', 'livingRooms', 'sale_time']
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X[cols_to_scale] = scaler_X.fit_transform(X[cols_to_scale])
X_test[cols_to_scale] = scaler_X.transform(X_test[cols_to_scale])
y = scaler_y.fit_transform(y.values.reshape(-1, 1))     # shape为(n, 1), transform要求为二阶张量

y = y.flatten()  # shape为(n,)
sigma = scaler_y.scale_[0]
mu = scaler_y.mean_[0]

数学推导:

目标是最小化$ \lvert e^{\sigma \hat{y} + \mu} - e^{\sigma y + \mu} \rvert $

等价于最小化 $ l(X, \beta) = \lvert e^{\sigma f(X, \beta)} - e^{\sigma y} \rvert $, 设其为损失函数

于是 $$ \frac{\partial l}{\partial f} = \sigma \cdot \operatorname{sign}(e^{\sigma f(X, \beta)} - e^{\sigma y}) \cdot e^{\sigma f(X, \beta)} $$
$$ \frac{\partial^2 l}{\partial f^2} = \sigma^2 \cdot \operatorname{sign}(e^{\sigma f} - e^{\sigma y}) \cdot e^{\sigma f} $$

In [ ]:
# XGBoost回归（随机搜索参数 + 自定义目标）
import xgboost as xgb
from sklearn.model_selection import ParameterSampler
from sklearn.model_selection import KFold

def exp_mae_obj(y_pred, dtrain):   # 需要数学推导
    y1 = np.exp(sigma * y_pred)
    y2 = np.exp(sigma * dtrain.get_label())

    grad = sigma * np.sign(y1 - y2) * y1
    hess = sigma ** 2 * y1   # xgb 要求 hess > 0, 故取绝对值近似
    return grad, hess

param_grid = {   # 只保留搜索后的最终结果
    'max_depth': [13],
    'eta': [0.01],   # 学习率
    'gamma': [0.1],
    'subsample': [0.9],
    'colsample_bytree': [0.9],
    'reg_alpha': [2],
    'reg_lambda': [0.1]
}

best_score = float('inf')
best_params = None
best_model = None

sampler = list(ParameterSampler(
    param_grid,
    n_iter = 2, 
    random_state = 42
))

for params in sampler:
    kf = KFold(
        n_splits = 5,
        shuffle = True,
        random_state = 42
    )
    fold_scores = []
    for fold, (train_idx, valid_idx) in enumerate(kf.split(X)):    
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y[train_idx], y[valid_idx]
        dtrain = xgb.DMatrix(X_train, y_train)
        dvalid = xgb.DMatrix(X_valid, y_valid)

        model = xgb.train(
            params,
            dtrain,
            num_boost_round = 500,
            obj = exp_mae_obj,
            verbose_eval = False,
            early_stopping_rounds = 20,
            evals = [(dvalid, 'eval')]   # 做早停和评估
        )

        y_pred = model.predict(dvalid)
        y1 = np.exp(sigma * y_pred)
        y2 = np.exp(sigma * dvalid.get_label())
        fold_score = np.mean(np.abs(y1 - y2))
        fold_scores.append(fold_score)

    score = np.mean(fold_scores)
    print(f"参数: {params}, MAE: {score:.4f}")
    if score < best_score:
        best_score = score
        best_params = params.copy()
        best_model = model

print("最优参数：", best_params)
print("最优分数：", best_score)

In [ ]:
# 多层感知机（网格超参数优化 + 自定义分数）
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer   # 自定义得分

def exp_mae(y_pred, y_true):
    y1 = np.exp(sigma * y_pred)
    y2 = np.exp(sigma * y_true)
    return np.mean(np.abs(y1 - y2))

my_scorer = make_scorer(exp_mae, greater_is_better = False)  # 越小越好

param_grid = {   # 只保留搜索后的最终结果
    'hidden_layer_sizes': [(128, 64, 32)],
    'activation': ['tanh'],
    'solver': ['adam'],
    'early_stopping': [True],
    'learning_rate_init': [0.001]
}

mlp = MLPRegressor(random_state = 42)
grid = GridSearchCV(
    mlp, 
    param_grid, 
    cv = 5, 
    scoring = my_scorer, 
    verbose = 3,
    n_jobs = -1
)
grid.fit(X, y)

print("最优参数:", grid.best_params_)
print("最优分数:", -grid.best_score_)

In [ ]:
# 寻找最佳集成权重
from sklearn.model_selection import train_test_split            
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size = 0.2, random_state = 68
)

params1 = {
    'max_depth': 13,
    'eta': 0.01,
    'gamma': 0.1,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'reg_alpha': 2,
    'reg_lambda': 0.1
}

params2 = {
    'hidden_layer_sizes': (128, 64, 32),
    'activation': 'tanh',
    'solver': 'adam',
    'early_stopping': True,
    'learning_rate_init': 0.001
}

pred1 = xgb.train(            
    params1,
    xgb.DMatrix(X_train, label = y_train),
    num_boost_round = 500,
    obj = exp_mae_obj
).predict(xgb.DMatrix(X_valid))

pred2 = MLPRegressor(
    **params2, 
).fit(X_train, y_train).predict(X_valid)

在 alpha 为0.0的情况下, 最后的 MAE 是0.43566145721051425
在 alpha 为0.1的情况下, 最后的 MAE 是0.4257276628157496
在 alpha 为0.2的情况下, 最后的 MAE 是0.4166797711791425
在 alpha 为0.3的情况下, 最后的 MAE 是0.4085588084231466
在 alpha 为0.4的情况下, 最后的 MAE 是0.4014621995464917
在 alpha 为0.5的情况下, 最后的 MAE 是0.3954686178605036
在 alpha 为0.6的情况下, 最后的 MAE 是0.3907345924174243
在 alpha 为0.7的情况下, 最后的 MAE 是0.387256818871441
在 alpha 为0.8的情况下, 最后的 MAE 是0.38513738412128784
在 alpha 为0.9的情况下, 最后的 MAE 是0.3842913319335405
在 alpha 为1.0的情况下, 最后的 MAE 是0.3848508723574558


In [ ]:
for num in range(11):
    alpha = num * 1.0 / 10
    pred = alpha * pred1 + (1 - alpha) * pred2
    mae = exp_mae(pred, y_valid)
    
    print(f"在 alpha 为{alpha}的情况下, 最后的 MAE 是{mae}")

alpha = 0.918
pred = alpha * pred1 + (1 - alpha) * pred2
mae = exp_mae(pred, y_valid)

print(f"在 alpha 为{alpha}的情况下, 最后的 MAE 是{mae}")

在 alpha 为0.0的情况下, 最后的 MAE 是0.43566145721051425
在 alpha 为0.1的情况下, 最后的 MAE 是0.4257276628157496
在 alpha 为0.2的情况下, 最后的 MAE 是0.4166797711791425
在 alpha 为0.3的情况下, 最后的 MAE 是0.4085588084231466
在 alpha 为0.4的情况下, 最后的 MAE 是0.4014621995464917
在 alpha 为0.5的情况下, 最后的 MAE 是0.3954686178605036
在 alpha 为0.6的情况下, 最后的 MAE 是0.3907345924174243
在 alpha 为0.7的情况下, 最后的 MAE 是0.387256818871441
在 alpha 为0.8的情况下, 最后的 MAE 是0.38513738412128784
在 alpha 为0.9的情况下, 最后的 MAE 是0.3842913319335405
在 alpha 为1.0的情况下, 最后的 MAE 是0.3848508723574558
在 alpha 为0.918的情况下, 最后的 MAE 是0.3842760966219636


In [46]:
# 最后模型
def reverse(y):   # 还原
    return np.exp(sigma * y + mu)

y_pred1 = xgb.train(            
    params1,
    xgb.DMatrix(X, label = y),
    num_boost_round = 500,
    obj = exp_mae_obj
).predict(xgb.DMatrix(X_test))

y_pred2 = MLPRegressor(
    **params2, 
).fit(X, y).predict(X_test)

y_pred = alpha * y_pred1 + (1 - alpha) * y_pred2
y_pred = reverse(y_pred)

In [47]:
# 提交
submission = pd.DataFrame({
    'ID': sample_submission['ID'],
    'price': y_pred
})

submission.to_csv('submission.csv', index = False)
print(submission)

           ID         price
0      266325  4.501637e+05
1      266326  4.068634e+05
2      266327  3.225658e+05
3      266328  4.609875e+05
4      266329  4.253614e+05
...       ...           ...
16542  282867  2.828450e+05
16543  282868  2.828450e+05
16544  282869  4.600934e+05
16545  282870  1.060476e+06
16546  282871  6.467526e+05

[16547 rows x 2 columns]
